# WIPS Reverse Problem Analysis
## Identifying Reporting Inconsistencies in OSHA Manufacturing Data

**Author:** Billy R. Davis | IRMB Research  
**Status:** Phase 2 Analysis — Follow-up to WIPS Preflight

### What This Notebook Does
Inverts the original WIPS prediction problem. Instead of
predicting which facilities are high risk, this analysis
predicts the expected injury rate for each facility
based on its operational characteristics and case detail
patterns, then flags facilities whose actual reported
rate diverges significantly from that expectation.

### Key Finding
260 facilities report injury rates significantly below
what their case detail patterns would predict. These
are candidates for closer review — not proven
underreporters.

### Critical Limitation
This analysis cannot distinguish between four possible
explanations for the gap: true safety performance,
definitional differences in reporting, timing
mismatches, or selective documentation. Only the last
constitutes actual underreporting. Public OSHA data
alone cannot resolve which explanation applies.

In [10]:
from google.colab import files
uploaded = files.upload()

Saving ITA_300A_Summary_Data_2024_through_12-31-2025.zip to ITA_300A_Summary_Data_2024_through_12-31-2025.zip


In [11]:
import pandas as pd
import numpy as np

# ─── Load 300A 2024 data ──────────────────────────
df_2024 = pd.read_csv(
    'ITA_300A_Summary_Data_2024_through_12-31-2025.zip',
    low_memory=False
)

# Filter manufacturing
mfg_2024 = df_2024[
    df_2024['naics_code'].astype(str).str.startswith(
        ('31','32','33'))
].copy()

# Calculate injury rate
mfg_2024['injury_rate'] = (
    mfg_2024['total_injuries'] /
    mfg_2024['annual_average_employees'].replace(0, 1)
) * 100

# Clean obvious data quality issues
mfg_clean = mfg_2024[
    (mfg_2024['injury_rate'] < 100) &
    (mfg_2024['annual_average_employees'] >= 5) &
    (mfg_2024['total_hours_worked'] > 0)
].copy()

# Risk tier classification
def classify_risk(rate):
    if rate == 0:
        return 0
    elif rate <= 3:
        return 1
    elif rate <= 10:
        return 2
    else:
        return 3

mfg_clean['risk_tier'] = mfg_clean['injury_rate'].apply(
    classify_risk)

print(f"Clean manufacturing records: {len(mfg_clean):,}")
print(f"Columns: {list(mfg_clean.columns)}")
print(f"\nInjury rate stats:")
print(mfg_clean['injury_rate'].describe().round(3))

# Save to Drive for next time
mfg_clean.to_csv(
    '/content/drive/MyDrive/wips_mfg_clean.csv',
    index=False
)
print("\nSaved to Drive: wips_mfg_clean.csv")

Clean manufacturing records: 60,702
Columns: ['id', 'establishment_name', 'establishment_id', 'ein', 'company_name', 'street_address', 'city', 'state', 'zip_code', 'naics_code', 'naics_year', 'industry_description', 'establishment_type', 'size', 'annual_average_employees', 'total_hours_worked', 'no_injuries_illnesses', 'total_deaths', 'total_dafw_cases', 'total_djtr_cases', 'total_other_cases', 'total_dafw_days', 'total_djtr_days', 'total_injuries', 'total_skin_disorders', 'total_respiratory_conditions', 'total_poisonings', 'total_hearing_loss', 'total_other_illnesses', 'created_timestamp', 'change_reason', 'year_filing_for', 'injury_rate', 'risk_tier']

Injury rate stats:
count    60702.000
mean         3.099
std          4.237
min          0.000
25%          0.000
50%          1.869
75%          4.348
max         86.667
Name: injury_rate, dtype: float64

Saved to Drive: wips_mfg_clean.csv


In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

features_reverse = [
    'naics_code',
    'size',
    'annual_average_employees',
    'total_hours_worked',
    'state',
    'year_filing_for'
]

X_rev = mfg_clean[features_reverse].copy()
y_rev = mfg_clean['injury_rate'].copy()

le_state_r = LabelEncoder()
le_naics_r = LabelEncoder()
X_rev['state'] = le_state_r.fit_transform(
    X_rev['state'].astype(str))
X_rev['naics_code'] = le_naics_r.fit_transform(
    X_rev['naics_code'].astype(str))

X_train_r, X_test_r, y_train_r, y_test_r = \
    train_test_split(X_rev, y_rev,
                     test_size=0.2, random_state=42)

reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

reg.fit(X_train_r, y_train_r)

mfg_clean['expected_rate'] = reg.predict(X_rev)
mfg_clean['actual_rate'] = mfg_clean['injury_rate']
mfg_clean['rate_gap'] = (
    mfg_clean['actual_rate'] -
    mfg_clean['expected_rate']
)

print("Expected vs Actual statistics:")
print(mfg_clean[['expected_rate',
                  'actual_rate',
                  'rate_gap']].describe().round(3))

Expected vs Actual statistics:
       expected_rate  actual_rate   rate_gap
count      60702.000    60702.000  60702.000
mean           3.120        3.099     -0.022
std            1.776        4.237      3.313
min            0.002        0.000    -18.153
25%            2.032        0.000     -1.948
50%            2.950        1.869     -0.633
75%            3.910        4.348      1.055
max           61.706       86.667     57.140


In [13]:
gap_std = mfg_clean['rate_gap'].std()
gap_mean = mfg_clean['rate_gap'].mean()
threshold = gap_mean - (1.5 * gap_std)

mfg_clean['underreport_flag'] = (
    mfg_clean['rate_gap'] < threshold
).astype(int)

flagged = mfg_clean[
    mfg_clean['underreport_flag'] == 1
].copy()

print(f"Total facilities: {len(mfg_clean):,}")
print(f"Underreporting candidates: {len(flagged):,}")
print(f"Percentage flagged: "
      f"{len(flagged)/len(mfg_clean)*100:.2f}%")

print(f"\nFlagged facility statistics:")
print(flagged[['expected_rate',
                'actual_rate',
                'rate_gap']].describe().round(3))

Total facilities: 60,702
Underreporting candidates: 553
Percentage flagged: 0.91%

Flagged facility statistics:
       expected_rate  actual_rate  rate_gap
count        553.000      553.000   553.000
mean           6.321        0.154    -6.167
std            1.796        0.890     1.519
min            4.992        0.000   -18.153
25%            5.230        0.000    -6.463
50%            5.640        0.000    -5.600
75%            6.592        0.000    -5.220
max           19.852       14.286    -4.992


In [14]:
print("Top NAICS codes among flagged facilities:")
print(flagged['industry_description']
      .value_counts().head(10))

print("\nSize distribution of flagged facilities:")
print(flagged['size'].value_counts().sort_index())

print("\nState distribution of flagged facilities:")
print(flagged['state'].value_counts().head(10))

print(f"\nAvg employees of flagged facilities:")
print(f"Mean: {flagged['annual_average_employees'].mean():.0f}")
print(f"Median: {flagged['annual_average_employees'].median():.0f}")

print(f"\nOverall manufacturing avg employees:")
print(f"Mean: {mfg_clean['annual_average_employees'].mean():.0f}")
print(f"Median: {mfg_clean['annual_average_employees'].median():.0f}")

Top NAICS codes among flagged facilities:
industry_description
Ready-mix concrete manufacturing and distributing                       28
Pallet parts, wood, manufacturing                                        9
Sawmills                                                                 8
Pallets, wood or wood and metal combination, manufacturing               8
Truck bodies assembling on purchased chassis                             7
Pallet containers, wood or wood and metal combination, manufacturing     7
Concrete batch plants (including temporary)                              7
Manufacturing                                                            6
Architectural wall panels, precast concrete, manufacturing               6
Buildings, prefabricated, wood, manufacturing                            5
Name: count, dtype: int64

Size distribution of flagged facilities:
size
1     173
2      22
3       4
21    319
22     35
Name: count, dtype: int64

State distribution of flagged facilit

In [15]:
# Check what we have in memory
print("Variables currently loaded:")
print(f"mfg_clean: {mfg_clean.shape if 'mfg_clean' in dir() else 'NOT LOADED'}")

# Check if the 301 case detail file is in Colab
import os
files_present = os.listdir('/content')
print(f"\nFiles in Colab:")
for f in files_present:
    if 'ITA' in f or 'OSHA' in f:
        print(f)

Variables currently loaded:
mfg_clean: (60702, 38)

Files in Colab:
ITA_300A_Summary_Data_2024_through_12-31-2025.zip


In [16]:
from google.colab import files
uploaded = files.upload()

Saving ITA_Case_Detail_Data_2024_through_12-31-2025.zip to ITA_Case_Detail_Data_2024_through_12-31-2025.zip


In [18]:
import pandas as pd
import numpy as np

# Load 301 case detail data
df_301 = pd.read_csv(
    'ITA_Case_Detail_Data_2024_through_12-31-2025.zip',
    low_memory=False,
    encoding='latin1'
)

# Filter to manufacturing
mfg_301 = df_301[
    df_301['naics_code'].astype(str).str.startswith(
        ('31','32','33'))
].copy()

print(f"301 manufacturing records: {len(mfg_301):,}")

# Aggregate to establishment level
from sklearn.preprocessing import LabelEncoder

# Encode the auto-coded label columns
for col in ['event_title_pred', 'nature_title_pred',
            'source_title_pred', 'part_title_pred']:
    mfg_301[col] = mfg_301[col].fillna('Unknown')
    le = LabelEncoder()
    mfg_301[col + '_enc'] = le.fit_transform(mfg_301[col])

# Aggregate per establishment
agg_301 = mfg_301.groupby('establishment_id').agg(
    total_cases=('incident_outcome', 'count'),
    avg_outcome=('incident_outcome', 'mean'),
    max_outcome=('incident_outcome', 'max'),
    most_common_event=('event_title_pred_enc',
                       lambda x: x.mode()[0]),
    most_common_nature=('nature_title_pred_enc',
                        lambda x: x.mode()[0]),
    most_common_source=('source_title_pred_enc',
                        lambda x: x.mode()[0])
).reset_index()

print(f"Establishments with case detail: {len(agg_301):,}")

# Merge with mfg_clean
mfg_full = mfg_clean.merge(
    agg_301, on='establishment_id', how='inner'
)

print(f"Merged records: {len(mfg_full):,}")
print(f"Columns: {list(mfg_full.columns)}")

301 manufacturing records: 104,631
Establishments with case detail: 11,023
Merged records: 11,013
Columns: ['id', 'establishment_name', 'establishment_id', 'ein', 'company_name', 'street_address', 'city', 'state', 'zip_code', 'naics_code', 'naics_year', 'industry_description', 'establishment_type', 'size', 'annual_average_employees', 'total_hours_worked', 'no_injuries_illnesses', 'total_deaths', 'total_dafw_cases', 'total_djtr_cases', 'total_other_cases', 'total_dafw_days', 'total_djtr_days', 'total_injuries', 'total_skin_disorders', 'total_respiratory_conditions', 'total_poisonings', 'total_hearing_loss', 'total_other_illnesses', 'created_timestamp', 'change_reason', 'year_filing_for', 'injury_rate', 'risk_tier', 'expected_rate', 'actual_rate', 'rate_gap', 'underreport_flag', 'total_cases', 'avg_outcome', 'max_outcome', 'most_common_event', 'most_common_nature', 'most_common_source']


## ⚠️ Methodology Note — Feature Selection
The v2 model uses 301 case detail features
(total_cases, avg_outcome, max_outcome) which are
derived from incidents that already occurred.

This is acceptable for the reverse problem because
we are not predicting future risk — we are checking
whether reported summary totals are internally
consistent with reported case detail patterns.

It would not be acceptable for the original
forward-prediction problem.

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

features_v2 = [
    'naics_code',
    'size',
    'annual_average_employees',
    'total_hours_worked',
    'state',
    'total_cases',
    'avg_outcome',
    'max_outcome',
    'most_common_event',
    'most_common_nature',
    'most_common_source'
]

X_v2 = mfg_full[features_v2].copy()
y_v2 = mfg_full['injury_rate'].copy()

le_s = LabelEncoder()
le_n = LabelEncoder()
X_v2['state'] = le_s.fit_transform(X_v2['state'].astype(str))
X_v2['naics_code'] = le_n.fit_transform(
    X_v2['naics_code'].astype(str))

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = \
    train_test_split(X_v2, y_v2,
                     test_size=0.2, random_state=42)

reg_v2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

reg_v2.fit(X_train_v2, y_train_v2)

mfg_full['expected_rate_v2'] = reg_v2.predict(X_v2)
mfg_full['rate_gap_v2'] = (
    mfg_full['injury_rate'] -
    mfg_full['expected_rate_v2']
)

print("Expected vs Actual statistics (v2):")
print(mfg_full[['expected_rate_v2',
                'injury_rate',
                'rate_gap_v2']].describe().round(3))

# Compare prediction spread
print(f"\nPrediction spread comparison:")
print(f"v1 expected std: {mfg_clean['expected_rate'].std():.3f}")
print(f"v2 expected std: {mfg_full['expected_rate_v2'].std():.3f}")
print(f"Actual rate std: {mfg_full['injury_rate'].std():.3f}")

Expected vs Actual statistics (v2):
       expected_rate_v2  injury_rate  rate_gap_v2
count         11013.000    11013.000    11013.000
mean              3.809        3.805       -0.003
std               3.220        3.456        0.785
min               0.298        0.000       -7.749
25%               1.625        1.562       -0.169
50%               2.830        2.842       -0.001
75%               4.849        4.965        0.088
max              38.144       50.000       25.450

Prediction spread comparison:
v1 expected std: 1.776
v2 expected std: 3.220
Actual rate std: 3.456


## Identifying Bidirectional Inconsistencies
The original reverse problem only flagged below-expected
facilities (potential underreporting). This expanded
version flags both directions:

- Below expected: facilities reporting fewer injuries
  than their case pattern suggests
- Above expected: facilities reporting more injuries
  than their case pattern suggests

Above-expected facilities may indicate definitional
issues, classification differences, or thorough
reporting. Both groups warrant review.

In [20]:
gap_std_v2 = mfg_full['rate_gap_v2'].std()
gap_mean_v2 = mfg_full['rate_gap_v2'].mean()

# Flag both directions this time
# Below expected = potential underreporting
# Above expected = unusually high vs case pattern
lower_threshold = gap_mean_v2 - (1.5 * gap_std_v2)
upper_threshold = gap_mean_v2 + (1.5 * gap_std_v2)

mfg_full['below_expected'] = (
    mfg_full['rate_gap_v2'] < lower_threshold
).astype(int)

mfg_full['above_expected'] = (
    mfg_full['rate_gap_v2'] > upper_threshold
).astype(int)

below = mfg_full[mfg_full['below_expected']==1].copy()
above = mfg_full[mfg_full['above_expected']==1].copy()

print(f"Total facilities analyzed: {len(mfg_full):,}")
print(f"\nBelow expected (potential under): {len(below):,}")
print(f"Above expected (rate spike vs case): {len(above):,}")
print(f"\nBelow group stats:")
print(below[['expected_rate_v2',
              'injury_rate',
              'rate_gap_v2']].describe().round(3))
print(f"\nAbove group stats:")
print(above[['expected_rate_v2',
              'injury_rate',
              'rate_gap_v2']].describe().round(3))

Total facilities analyzed: 11,013

Below expected (potential under): 260
Above expected (rate spike vs case): 328

Below group stats:
       expected_rate_v2  injury_rate  rate_gap_v2
count           260.000      260.000      260.000
mean              5.832        3.732       -2.099
std               4.050        3.912        1.103
min               1.188        0.000       -7.749
25%               2.599        0.627       -2.400
50%               4.749        2.667       -1.741
75%               7.523        5.776       -1.403
max              22.473       20.183       -1.182

Above group stats:
       expected_rate_v2  injury_rate  rate_gap_v2
count           328.000      328.000      328.000
mean              7.277       10.011        2.734
std               6.808        7.789        2.371
min               0.386        2.203        1.177
25%               2.309        4.775        1.500
50%               4.765        7.240        2.058
75%               9.975       12.840        2.

In [21]:
print("═" * 50)
print("BELOW EXPECTED — Potential Underreporting")
print("═" * 50)
print("\nTop industries:")
print(below['industry_description']
      .value_counts().head(10))
print(f"\nAvg employees: {below['annual_average_employees'].mean():.0f}")
print(f"Median employees: {below['annual_average_employees'].median():.0f}")

print("\n" + "═" * 50)
print("ABOVE EXPECTED — Rate Higher Than Case Pattern")
print("═" * 50)
print("\nTop industries:")
print(above['industry_description']
      .value_counts().head(10))
print(f"\nAvg employees: {above['annual_average_employees'].mean():.0f}")
print(f"Median employees: {above['annual_average_employees'].median():.0f}")

print("\n" + "═" * 50)
print("OVERALL MFG_FULL POPULATION")
print("═" * 50)
print(f"\nAvg employees: {mfg_full['annual_average_employees'].mean():.0f}")
print(f"Median employees: {mfg_full['annual_average_employees'].median():.0f}")

══════════════════════════════════════════════════
BELOW EXPECTED — Potential Underreporting
══════════════════════════════════════════════════

Top industries:
industry_description
Pallet parts, wood, manufacturing                                7
Truss Manufacturing                                              5
Poultry Feed Mill                                                4
Shipyard (i.e., facility capable of building ships)              3
Granulated beet sugar manufacturing                              3
Machine shops                                                    3
Cheese (except cottage cheese) manufacturing                     3
Assembly plants, passenger car, on chassis of own manufacture    3
Trusses, wood roof or floor, manufacturing                       3
Anodizing metals and metal products for the trade                3
Name: count, dtype: int64

Avg employees: 1027
Median employees: 130

══════════════════════════════════════════════════
ABOVE EXPECTED — Rate Highe

## Analysis Results

| Group | Count | Mean Gap |
|---|---|---|
| Below expected | 260 facilities | -2.10 |
| Above expected | 328 facilities | +2.73 |

### Industry Patterns
**Below expected clusters in:**
- Pallet and wood manufacturing
- Trusses and structural lumber
- Shipyards and machine shops

**Above expected clusters in:**
- Soft drink and beverage manufacturing
- Commercial bakeries
- Foundries

### What This Means
The asymmetry between physical labor industries
(below expected) and food/beverage manufacturing
(above expected) is worth investigating further but
cannot be explained from public data alone.

### Next Phase
External validation against independent data sources
including BLS injury rates, state workers compensation
data, and public OSHA citation records.

---
*IRMB Research | Billy R. Davis | Romans 8:28*